In [1]:
#!/usr/bin/env python3
"""
Beat-aligned dataset builder for Taiko chart generation.

This script performs the following workflow:
1. Traverse the unpacked dataset folder and generate an internal chart mapping table.
2. Parse timing information from each timing.json.
3. Compute beat-grid information.
4. Parse and analyze note events.
5. Build a beat-aligned frame timeline.
6. Build a raw mel spectrogram from the audio file.
7. Interpolate the raw mel spectrogram onto the beat-aligned frame timeline.
8. Segment the aligned mel spectrogram into 4-beat sequences.
9. Build per-sequence event tokens.

Outputs are written into two top-level folders under the data root:
- chart_index
- beat_aligned_dataset

The script is designed for GitHub use:
- clear English comments
- modular functions
- conservative logging
- error handling at both chart and pipeline levels
"""

from __future__ import annotations

import json
import logging
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import librosa
import numpy as np
import pandas as pd

C:\Users\Xiang Gao\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# ============================================================================
# Default paths and global configuration
# ============================================================================

DEFAULT_DATA_ROOT = Path(r"D:\Study Abroad\course\DSCI498\Project\data")
DEFAULT_UNPACKED_ROOT = DEFAULT_DATA_ROOT / "unpacked"
DEFAULT_INDEX_DIR = DEFAULT_DATA_ROOT / "chart_index"
DEFAULT_DATASET_DIR = DEFAULT_DATA_ROOT / "beat_aligned_dataset"

FRAMES_PER_BEAT = 48
SEQUENCE_BEATS = 4
FRAMES_PER_SEQUENCE = FRAMES_PER_BEAT * SEQUENCE_BEATS

N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128

MODEL_EVENT_TYPES = {
    "don",
    "kat",
    "bigdon",
    "bigkat",
    "drumroll",
    "sliderstart",
    "sliderend",
}

ALLOWED_EVENT_TYPES = MODEL_EVENT_TYPES | {"bpmchange"}


In [3]:
@dataclass
class TimingInfo:
    offset_ms: float
    beat_duration_ms: float
    bpm: float
    meter: int
    n_bpm_points: int
    n_timing_points: int


@dataclass
class BeatGridInfo:
    total_beats: int
    total_frames: int
    total_sequences: int
    frame_duration_ms: float
    last_beat_time_ms: float
    last_frame_time_ms: float
    remaining_tail_ms: float
    frame_overshoot_ms: float


@dataclass
class NotesInfo:
    total_events: int
    model_events: int
    unknown_event_types: List[str]
    min_model_frame: Optional[int]
    max_model_frame: Optional[int]
    outside_event_count: int
    collision_frame_count: int
    collision_event_total: int
    n_at_frame0: int
    n_at_last_frame: int
    event_type_counts: Dict[str, int]


In [4]:
# ============================================================================
# Utility helpers
# ============================================================================


def setup_logging() -> None:
    """Configure concise logging for command-line execution."""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%H:%M:%S",
    )



def ensure_dir(path: Path) -> None:
    """Create a directory if it does not already exist."""
    path.mkdir(parents=True, exist_ok=True)



def sanitize_filename(text: str, max_length: int = 150) -> str:
    """Convert an arbitrary chart name into a filesystem-safe stem."""
    safe = re.sub(r"[^A-Za-z0-9._-]+", "_", text).strip("._-")
    if not safe:
        safe = "chart"
    return safe[:max_length]



def chart_uid(folder_id: Any, chart_base: str) -> str:
    """Build a stable chart identifier for saved outputs."""
    return f"{folder_id}_{sanitize_filename(chart_base)}"



def safe_json_dump(obj: Any, path: Path) -> None:
    """Write JSON with UTF-8 encoding and readable indentation."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)



def require_file(path: Path, label: str) -> None:
    """Raise a clear error if a required file is missing."""
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")

In [5]:
# ============================================================================
# Step 1: build mapping table by traversing the unpacked folder
# ============================================================================


def build_chart_mapping_table(unpacked_root: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Traverse the unpacked dataset directory and build a chart-level mapping table.

    Expected folder structure:
        unpacked/<folder_id>/
            <audio>.mp3 or <audio>.ogg
            parsed/
                <chart_base>.notes.json
                <chart_base>.timing.json
                <chart_base>.metadata.json

    Returns
    -------
    mapping_df:
        One row per complete chart triple.
    issues_df:
        Folder-level issues such as missing audio, multiple audio files,
        or incomplete chart triples.
    """
    rows: List[Dict[str, Any]] = []
    issues: List[Dict[str, Any]] = []

    if not unpacked_root.exists():
        raise FileNotFoundError(f"Unpacked root not found: {unpacked_root}")

    folder_paths = sorted([p for p in unpacked_root.iterdir() if p.is_dir()])
    logging.info("Scanning unpacked folders: %d", len(folder_paths))

    for folder_path in folder_paths:
        parsed_path = folder_path / "parsed"
        folder_id = folder_path.name

        audio_files = sorted(list(folder_path.glob("*.mp3")) + list(folder_path.glob("*.ogg")))
        if len(audio_files) != 1:
            issues.append(
                {
                    "folder_id": folder_id,
                    "folder_path": str(folder_path),
                    "issue_type": "audio_count_error",
                    "issue_detail": f"Expected 1 audio file, found {len(audio_files)}",
                }
            )
            continue

        audio_path = audio_files[0]

        if not parsed_path.exists():
            issues.append(
                {
                    "folder_id": folder_id,
                    "folder_path": str(folder_path),
                    "issue_type": "missing_parsed_folder",
                    "issue_detail": "parsed/ folder not found",
                }
            )
            continue

        notes_files = sorted(parsed_path.glob("*.notes.json"))
        timing_files = sorted(parsed_path.glob("*.timing.json"))
        metadata_files = sorted(parsed_path.glob("*.metadata.json"))

        notes_map = {p.name[:-11]: p for p in notes_files}      # strip '.notes.json'
        timing_map = {p.name[:-12]: p for p in timing_files}    # strip '.timing.json'
        metadata_map = {p.name[:-14]: p for p in metadata_files}  # strip '.metadata.json'

        chart_bases = sorted(set(notes_map) | set(timing_map) | set(metadata_map))

        if not chart_bases:
            issues.append(
                {
                    "folder_id": folder_id,
                    "folder_path": str(folder_path),
                    "issue_type": "no_parsed_charts",
                    "issue_detail": "No parsed chart files found",
                }
            )
            continue

        for base in chart_bases:
            notes_path = notes_map.get(base)
            timing_path = timing_map.get(base)
            metadata_path = metadata_map.get(base)

            if not (notes_path and timing_path and metadata_path):
                issues.append(
                    {
                        "folder_id": folder_id,
                        "folder_path": str(folder_path),
                        "issue_type": "incomplete_chart_triple",
                        "issue_detail": (
                            f"chart_base={base}; "
                            f"notes={notes_path is not None}; "
                            f"timing={timing_path is not None}; "
                            f"metadata={metadata_path is not None}"
                        ),
                    }
                )
                continue

            rows.append(
                {
                    "folder_id": folder_id,
                    "folder_path": str(folder_path),
                    "audio_file": audio_path.name,
                    "audio_path": str(audio_path),
                    "chart_base": base,
                    "notes_path": str(notes_path),
                    "timing_path": str(timing_path),
                    "metadata_path": str(metadata_path),
                }
            )

    mapping_df = pd.DataFrame(rows)
    issues_df = pd.DataFrame(issues)
    return mapping_df, issues_df

In [6]:
# ============================================================================
# Step 2: timing information
# ============================================================================


def get_timing_info(timing_path: Path) -> TimingInfo:
    """
    Parse a timing.json file and extract constant-BPM timing information.

    Raises
    ------
    ValueError
        If timing points are missing or if the chart contains BPM changes.
    """
    require_file(timing_path, "timing.json")

    with open(timing_path, "r", encoding="utf-8") as f:
        timing_data = json.load(f)

    timing_points = timing_data.get("timing_points", [])
    if not timing_points:
        raise ValueError("No timing_points found")

    bpm_points = [tp for tp in timing_points if int(tp.get("uninherited", 0)) == 1]
    if not bpm_points:
        raise ValueError("No BPM timing points found (uninherited=1)")

    bpm_points = sorted(bpm_points, key=lambda x: float(x["offset"]))
    unique_ms_per_beat = sorted({round(float(tp["ms_per_beat"]), 10) for tp in bpm_points})

    if len(unique_ms_per_beat) != 1:
        raise ValueError(f"Non-constant BPM detected: {unique_ms_per_beat}")

    beat_duration_ms = float(bpm_points[0]["ms_per_beat"])
    offset_ms = float(bpm_points[0]["offset"])
    meter = int(bpm_points[0].get("meter", 4))
    bpm = 60000.0 / beat_duration_ms

    return TimingInfo(
        offset_ms=offset_ms,
        beat_duration_ms=beat_duration_ms,
        bpm=bpm,
        meter=meter,
        n_bpm_points=len(bpm_points),
        n_timing_points=len(timing_points),
    )


In [7]:
# ============================================================================
# Audio helper
# ============================================================================


def get_audio_info(audio_path: Path) -> Dict[str, Any]:
    """Load audio and return waveform plus basic audio metadata."""
    require_file(audio_path, "audio file")

    y, sr = librosa.load(audio_path, sr=None, mono=True)
    n_samples = len(y)
    audio_duration_sec = n_samples / sr
    audio_duration_ms = audio_duration_sec * 1000.0

    return {
        "waveform": y,
        "sample_rate": sr,
        "n_samples": n_samples,
        "audio_duration_sec": audio_duration_sec,
        "audio_duration_ms": audio_duration_ms,
    }

In [8]:
# ============================================================================
# Step 3: beat grid information
# ============================================================================

def compute_beat_grid_info(
    offset_ms: float,
    beat_duration_ms: float,
    audio_duration_ms: float,
) -> Tuple[BeatGridInfo, np.ndarray]:
    """
    Compute beat times and beat-grid summary values for one chart.

    The beat grid starts from the first BPM timing point offset and expands
    forward until the audio duration is reached.
    """
    beat_times_ms: List[float] = []
    t = offset_ms
    while t < audio_duration_ms:
        beat_times_ms.append(t)
        t += beat_duration_ms

    if not beat_times_ms:
        raise ValueError("No beat times generated; check offset and audio duration")

    beat_times_ms_arr = np.array(beat_times_ms, dtype=np.float64)
    total_beats = len(beat_times_ms_arr)
    total_frames = total_beats * FRAMES_PER_BEAT
    total_sequences = total_frames // FRAMES_PER_SEQUENCE
    frame_duration_ms = beat_duration_ms / FRAMES_PER_BEAT
    last_frame_time_ms = offset_ms + (total_frames - 1) * frame_duration_ms

    info = BeatGridInfo(
        total_beats=total_beats,
        total_frames=total_frames,
        total_sequences=total_sequences,
        frame_duration_ms=frame_duration_ms,
        last_beat_time_ms=float(beat_times_ms_arr[-1]),
        last_frame_time_ms=float(last_frame_time_ms),
        remaining_tail_ms=float(audio_duration_ms - beat_times_ms_arr[-1]),
        frame_overshoot_ms=float(last_frame_time_ms - audio_duration_ms),
    )
    return info, beat_times_ms_arr

In [9]:
MODEL_EVENT_TYPES: Set[str] = {
    "don", "kat", "bigdon", "bigkat",
    "drumroll", "sliderstart", "sliderend"
}

ALL_EXPECTED_EVENT_TYPES: Set[str] = {
    "don", "kat", "bigdon", "bigkat",
    "drumroll", "sliderstart", "sliderend", "bpmchange"
}


def load_note_events(notes_path: Path) -> List[Dict[str, Any]]:
    """
    Load note events from notes.json, supporting common top-level formats.

    Supported top-level structures:
    - list[dict]
    - {"notes": [...]}
    - {"events": [...]}
    - {"hit_objects": [...]}
    """
    require_file(notes_path, "notes.json")

    with open(notes_path, "r", encoding="utf-8") as f:
        notes_data = json.load(f)

    if isinstance(notes_data, list):
        events = notes_data
    elif isinstance(notes_data, dict):
        if "notes" in notes_data:
            events = notes_data["notes"]
        elif "events" in notes_data:
            events = notes_data["events"]
        elif "hit_objects" in notes_data:
            events = notes_data["hit_objects"]
        else:
            raise ValueError(f"Unknown notes.json structure. Keys: {list(notes_data.keys())}")
    else:
        raise ValueError("Unsupported notes.json structure")

    if not events:
        raise ValueError("No events found in notes.json")

    return events

In [10]:
def compute_notes_info(
    events: Sequence[Dict[str, Any]],
    offset_ms: float,
    beat_duration_ms: float,
    total_frames: int,
) -> Tuple[NotesInfo, pd.DataFrame, pd.DataFrame]:
    """
    Convert note times into beat-relative frame coordinates and summarize them.

    Returns
    -------
    notes_info:
        Chart-level summary of event quality and counts.
    events_df:
        All events mapped to frame coordinates.
    model_df:
        Only modeling events mapped to frame coordinates.
    """
    events_df = pd.DataFrame(events).copy()
    if "type" not in events_df.columns or "time" not in events_df.columns:
        raise ValueError(f"Required event fields missing. Columns: {list(events_df.columns)}")

    events_df["time"] = events_df["time"].astype(float)
    events_df["beat_position"] = (events_df["time"] - offset_ms) / beat_duration_ms
    events_df["frame_position"] = events_df["beat_position"] * FRAMES_PER_BEAT
    events_df["frame_index_rounded"] = events_df["frame_position"].round().astype(int)

    event_type_counts = events_df["type"].value_counts().to_dict()
    unknown_event_types = sorted(set(events_df["type"]) - ALLOWED_EVENT_TYPES)

    model_df = events_df[events_df["type"].isin(MODEL_EVENT_TYPES)].copy()

    # ---------------------------------------------------------------------
    # Boundary handling
    # ---------------------------------------------------------------------
    # 1. Tolerate tiny negative drift near frame 0.
    #    Example: a note slightly earlier than offset due to parser rounding.
    small_negative_mask = (
        (model_df["frame_index_rounded"] < 0)
        & (model_df["frame_position"] >= -0.5)
    )
    model_df.loc[small_negative_mask, "frame_index_rounded"] = 0

    # 2. Drop tail-overflow events beyond the valid frame grid.
    #    These usually occur at the very end and are not critical for learning.
    model_df = model_df[model_df["frame_index_rounded"] >= 0].copy()
    model_df = model_df[model_df["frame_index_rounded"] < total_frames].copy()

    # 3. After the tolerance step above, any remaining negative frames are
    #    considered true invalid events.
    negative_outside_df = model_df[model_df["frame_index_rounded"] < 0]
    if not negative_outside_df.empty:
        raise ValueError(f"Found {len(negative_outside_df)} note events before frame 0")

    model_df = model_df.sort_values(["frame_index_rounded", "time"]).reset_index(drop=True)

    outside_df = model_df[
        (model_df["frame_index_rounded"] < 0)
        | (model_df["frame_index_rounded"] >= total_frames)
    ]

    frame_counts = model_df["frame_index_rounded"].value_counts()
    collision_frames = frame_counts[frame_counts > 1]

    min_model_frame = int(model_df["frame_index_rounded"].min()) if not model_df.empty else None
    max_model_frame = int(model_df["frame_index_rounded"].max()) if not model_df.empty else None
    n_at_frame0 = int((model_df["frame_index_rounded"] == 0).sum()) if not model_df.empty else 0
    n_at_last_frame = int((model_df["frame_index_rounded"] == total_frames - 1).sum()) if not model_df.empty else 0

    notes_info = NotesInfo(
        total_events=int(len(events_df)),
        model_events=int(len(model_df)),
        unknown_event_types=unknown_event_types,
        min_model_frame=min_model_frame,
        max_model_frame=max_model_frame,
        outside_event_count=int(len(outside_df)),
        collision_frame_count=int(len(collision_frames)),
        collision_event_total=int(collision_frames.sum()) if not collision_frames.empty else 0,
        n_at_frame0=n_at_frame0,
        n_at_last_frame=n_at_last_frame,
        event_type_counts={str(k): int(v) for k, v in event_type_counts.items()},
    )
    return notes_info, events_df, model_df

In [11]:
# ============================================================================
# Step 5: beat-aligned frame timeline
# ============================================================================


def build_beat_aligned_frame_timeline(
    offset_ms: float,
    beat_duration_ms: float,
    total_frames: int,
) -> np.ndarray:
    """
    Build the target frame timeline where 1 beat = 48 frames.

    The resulting array has length total_frames and stores the real time (ms)
    corresponding to each beat-aligned frame.
    """
    frame_duration_ms = beat_duration_ms / FRAMES_PER_BEAT
    frame_times_ms = offset_ms + np.arange(total_frames, dtype=np.float64) * frame_duration_ms
    return frame_times_ms

In [12]:
# ============================================================================
# Step 6: raw mel spectrogram
# ============================================================================


def build_raw_mel_spectrogram(
    waveform: np.ndarray,
    sample_rate: int,
    n_fft: int = N_FFT,
    hop_length: int = HOP_LENGTH,
    n_mels: int = N_MELS,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a conventional mel spectrogram and its original time axis.

    Returns
    -------
    mel_spec_db:
        Shape = (n_original_frames, n_mels)
    orig_frame_times_ms:
        Shape = (n_original_frames,)
    """
    mel_spec = librosa.feature.melspectrogram(
        y=waveform,
        sr=sample_rate,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0,
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max).T.astype(np.float32)

    orig_frame_times_sec = librosa.frames_to_time(
        np.arange(mel_spec_db.shape[0]),
        sr=sample_rate,
        hop_length=hop_length,
        n_fft=n_fft,
    )
    orig_frame_times_ms = orig_frame_times_sec * 1000.0
    return mel_spec_db, orig_frame_times_ms

In [13]:
# ============================================================================
# Step 7: interpolation onto beat-aligned timeline
# ============================================================================


def interpolate_raw_mel_to_beat_aligned_timeline(
    mel_spec_db: np.ndarray,
    orig_frame_times_ms: np.ndarray,
    beat_aligned_frame_times_ms: np.ndarray,
) -> np.ndarray:
    """
    Interpolate the raw mel spectrogram onto the beat-aligned frame timeline.

    Edge behavior is intentionally conservative:
    - values before the first raw frame use the first raw frame
    - values after the last raw frame use the last raw frame
    """
    if mel_spec_db.ndim != 2:
        raise ValueError(f"Expected 2D mel spectrogram, got shape {mel_spec_db.shape}")

    n_target_frames = len(beat_aligned_frame_times_ms)
    n_mels = mel_spec_db.shape[1]
    aligned = np.empty((n_target_frames, n_mels), dtype=np.float32)

    for mel_bin in range(n_mels):
        aligned[:, mel_bin] = np.interp(
            beat_aligned_frame_times_ms,
            orig_frame_times_ms,
            mel_spec_db[:, mel_bin],
            left=float(mel_spec_db[0, mel_bin]),
            right=float(mel_spec_db[-1, mel_bin]),
        )

    if np.isnan(aligned).any():
        raise ValueError("NaN detected after interpolation")

    return aligned

In [14]:
# ============================================================================
# Step 8: segment aligned mel into 4-beat sequences
# ============================================================================


def segment_aligned_mel_into_4beat_sequences(
    aligned_mel_db: np.ndarray,
    total_sequences: int,
) -> np.ndarray:
    """
    Segment the aligned mel spectrogram into 4-beat windows.

    Output shape:
        (n_sequences, 192, n_mels)
    """
    if aligned_mel_db.shape[0] < total_sequences * FRAMES_PER_SEQUENCE:
        raise ValueError(
            "Aligned mel spectrogram is shorter than the expected number of full sequences"
        )

    sequences = []
    for seq_idx in range(total_sequences):
        start_frame = seq_idx * FRAMES_PER_SEQUENCE
        end_frame = start_frame + FRAMES_PER_SEQUENCE
        segment = aligned_mel_db[start_frame:end_frame]
        if segment.shape[0] != FRAMES_PER_SEQUENCE:
            raise ValueError(
                f"Unexpected sequence length at seq_idx={seq_idx}: {segment.shape}"
            )
        sequences.append(segment)

    if not sequences:
        raise ValueError("No full 4-beat sequences were created")

    return np.stack(sequences, axis=0).astype(np.float32)

In [15]:
# ============================================================================
# Step 9: per-sequence event tokens
# ============================================================================


def build_per_sequence_event_tokens(model_df: pd.DataFrame, total_sequences: int) -> List[Dict[str, Any]]:
    """
    Build local token sequences for each 4-beat segment.

    Token rule:
        time shift is measured in beat-aligned frames within the current sequence
        and represented as TS_<delta_frames>.
    """
    token_data: List[Dict[str, Any]] = []

    for seq_idx in range(total_sequences):
        seq_start_frame = seq_idx * FRAMES_PER_SEQUENCE
        seq_end_frame = seq_start_frame + FRAMES_PER_SEQUENCE - 1

        seq_events = model_df[
            (model_df["frame_index_rounded"] >= seq_start_frame)
            & (model_df["frame_index_rounded"] <= seq_end_frame)
        ].copy()
        seq_events["local_frame"] = seq_events["frame_index_rounded"] - seq_start_frame
        seq_events = seq_events.sort_values(["local_frame", "time"]).reset_index(drop=True)

        tokens: List[str] = []
        prev_local_frame = 0
        for _, row in seq_events.iterrows():
            local_frame = int(row["local_frame"])
            event_type = str(row["type"]).upper()
            time_shift = local_frame - prev_local_frame
            if time_shift > 0:
                tokens.append(f"TS_{time_shift}")
            tokens.append(event_type)
            prev_local_frame = local_frame

        token_data.append(
            {
                "seq_idx": seq_idx,
                "start_frame": seq_start_frame,
                "end_frame": seq_end_frame,
                "n_events": int(len(seq_events)),
                "n_tokens": int(len(tokens)),
                "tokens": tokens,
            }
        )

    return token_data

In [16]:
# ============================================================================
# Wrapper for one chart
# ============================================================================


def process_one_chart_row(row: pd.Series, dataset_dir: Path) -> Dict[str, Any]:
    """
    Run the full 9-step pipeline for one chart row from the mapping table.

    The function writes per-chart outputs to disk and returns a compact summary.
    """
    folder_id = row["folder_id"]
    chart_base = row["chart_base"]
    chart_id = chart_uid(folder_id, chart_base)

    audio_path = Path(row["audio_path"])
    notes_path = Path(row["notes_path"])
    timing_path = Path(row["timing_path"])
    metadata_path = Path(row["metadata_path"])

    metadata: Dict[str, Any] = {}
    if metadata_path.exists():
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

    timing_info = get_timing_info(timing_path)
    audio_info = get_audio_info(audio_path)
    beat_grid_info, _ = compute_beat_grid_info(
        timing_info.offset_ms,
        timing_info.beat_duration_ms,
        audio_info["audio_duration_ms"],
    )
    events = load_note_events(notes_path)
    notes_info, _events_df, model_df = compute_notes_info(
        events,
        timing_info.offset_ms,
        timing_info.beat_duration_ms,
        beat_grid_info.total_frames,
    )

    if notes_info.model_events == 0:
        raise ValueError("No modeling events found after filtering event types")
    if notes_info.outside_event_count > 0:
        raise ValueError(f"Found {notes_info.outside_event_count} note events outside frame grid")
    if notes_info.collision_frame_count > 0:
        raise ValueError(f"Found {notes_info.collision_frame_count} collision frames")
    if beat_grid_info.total_sequences == 0:
        raise ValueError("No full 4-beat sequences available")

    frame_times_ms = build_beat_aligned_frame_timeline(
        timing_info.offset_ms,
        timing_info.beat_duration_ms,
        beat_grid_info.total_frames,
    )
    mel_spec_db, orig_frame_times_ms = build_raw_mel_spectrogram(
        audio_info["waveform"],
        audio_info["sample_rate"],
    )
    aligned_mel_db = interpolate_raw_mel_to_beat_aligned_timeline(
        mel_spec_db,
        orig_frame_times_ms,
        frame_times_ms,
    )
    audio_sequences = segment_aligned_mel_into_4beat_sequences(
        aligned_mel_db,
        beat_grid_info.total_sequences,
    )
    token_data = build_per_sequence_event_tokens(model_df, beat_grid_info.total_sequences)

    audio_npz_dir = dataset_dir / "audio_npz"
    token_json_dir = dataset_dir / "token_json"
    ensure_dir(audio_npz_dir)
    ensure_dir(token_json_dir)

    np.savez_compressed(
        audio_npz_dir / f"{chart_id}.npz",
        audio_sequences=audio_sequences,
    )
    safe_json_dump(token_data, token_json_dir / f"{chart_id}.json")

    sequence_metadata = []
    for seq in token_data:
        sequence_metadata.append(
            {
                "chart_id": chart_id,
                "folder_id": folder_id,
                "chart_base": chart_base,
                "seq_idx": seq["seq_idx"],
                "start_frame": seq["start_frame"],
                "end_frame": seq["end_frame"],
                "n_events": seq["n_events"],
                "n_tokens": seq["n_tokens"],
                "audio_npz_path": str(audio_npz_dir / f"{chart_id}.npz"),
                "token_json_path": str(token_json_dir / f"{chart_id}.json"),
            }
        )

    summary = {
        "chart_id": chart_id,
        "folder_id": folder_id,
        "chart_base": chart_base,
        "title": metadata.get("title", ""),
        "artist": metadata.get("artist", ""),
        "difficulty": metadata.get("difficulty", ""),
        "mode": metadata.get("mode", ""),
        "status": "ok",
        "error_message": "",
        "audio_path": str(audio_path),
        "notes_path": str(notes_path),
        "timing_path": str(timing_path),
        "metadata_path": str(metadata_path),
        "offset_ms": timing_info.offset_ms,
        "beat_duration_ms": timing_info.beat_duration_ms,
        "bpm": timing_info.bpm,
        "meter": timing_info.meter,
        "audio_duration_ms": audio_info["audio_duration_ms"],
        "total_beats": beat_grid_info.total_beats,
        "total_frames": beat_grid_info.total_frames,
        "total_sequences": beat_grid_info.total_sequences,
        "frame_overshoot_ms": beat_grid_info.frame_overshoot_ms,
        "total_events": notes_info.total_events,
        "model_events": notes_info.model_events,
        "outside_event_count": notes_info.outside_event_count,
        "collision_frame_count": notes_info.collision_frame_count,
        "unknown_event_types": "|".join(notes_info.unknown_event_types),
        "audio_sequences_shape": str(tuple(audio_sequences.shape)),
    }
    return {
        "summary": summary,
        "sequence_metadata": sequence_metadata,
    }

In [17]:
def run_pipeline(
    unpacked_root: Path = DEFAULT_UNPACKED_ROOT,
    index_dir: Path = DEFAULT_INDEX_DIR,
    dataset_dir: Path = DEFAULT_DATASET_DIR,
) -> None:
    """
    End-to-end execution:
    1. Build internal mapping table.
    2. Save mapping and mapping issues.
    3. Process every chart row in the mapping table.
    4. Save chart-level and sequence-level summaries.
    """
    ensure_dir(index_dir)
    ensure_dir(dataset_dir)

    logging.info("Building internal chart mapping table...")
    mapping_df, issues_df = build_chart_mapping_table(unpacked_root)

    mapping_csv = index_dir / "audio_chart_mapping_generated.csv"
    issues_csv = index_dir / "mapping_issues.csv"
    mapping_df.to_csv(mapping_csv, index=False, encoding="utf-8-sig")
    issues_df.to_csv(issues_csv, index=False, encoding="utf-8-sig")

    logging.info("Mapping rows generated: %d", len(mapping_df))
    logging.info("Mapping issues found: %d", len(issues_df))

    if mapping_df.empty:
        raise RuntimeError("No valid chart mapping rows were created")

    chart_summaries: List[Dict[str, Any]] = []
    sequence_metadata_rows: List[Dict[str, Any]] = []

    total_rows = len(mapping_df)
    for i, (_, row) in enumerate(mapping_df.iterrows(), start=1):
        chart_label = f"folder_id={row['folder_id']} | chart={row['chart_base']}"
        if i == 1 or i == total_rows or i % 20 == 0:
            logging.info("Processing %d / %d | %s", i, total_rows, chart_label)

        try:
            result = process_one_chart_row(row, dataset_dir)
            chart_summaries.append(result["summary"])
            sequence_metadata_rows.extend(result["sequence_metadata"])
        except Exception as exc:
            logging.error("Failed on %s | %s", chart_label, exc)
            chart_summaries.append(
                {
                    "chart_id": chart_uid(row["folder_id"], row["chart_base"]),
                    "folder_id": row["folder_id"],
                    "chart_base": row["chart_base"],
                    "status": "error",
                    "error_message": str(exc),
                    "audio_path": row["audio_path"],
                    "notes_path": row["notes_path"],
                    "timing_path": row["timing_path"],
                    "metadata_path": row["metadata_path"],
                }
            )
            continue

    chart_summary_df = pd.DataFrame(chart_summaries)
    sequence_metadata_df = pd.DataFrame(sequence_metadata_rows)

    chart_summary_csv = index_dir / "chart_build_summary.csv"
    sequence_metadata_csv = dataset_dir / "sequence_metadata.csv"
    chart_summary_df.to_csv(chart_summary_csv, index=False, encoding="utf-8-sig")
    sequence_metadata_df.to_csv(sequence_metadata_csv, index=False, encoding="utf-8-sig")

    ok_count = int((chart_summary_df["status"] == "ok").sum()) if not chart_summary_df.empty else 0
    error_count = int((chart_summary_df["status"] == "error").sum()) if not chart_summary_df.empty else 0

    logging.info("Pipeline finished")
    logging.info("Charts succeeded: %d", ok_count)
    logging.info("Charts failed: %d", error_count)
    logging.info("Mapping CSV saved to: %s", mapping_csv)
    logging.info("Chart summary saved to: %s", chart_summary_csv)
    logging.info("Sequence metadata saved to: %s", sequence_metadata_csv)

In [18]:
setup_logging()
run_pipeline()

19:42:37 | INFO | Building internal chart mapping table...
19:42:37 | INFO | Scanning unpacked folders: 102
19:42:37 | INFO | Mapping rows generated: 290
19:42:37 | INFO | Mapping issues found: 19
19:42:37 | INFO | Processing 1 / 290 | folder_id=1024653 | chart=Mizukara o Enshutsu Suru Otome no Kai - Girlish Lover (yuki_momoiro722) [Genjuro's Futsuu]
19:42:57 | INFO | Processing 20 / 290 | folder_id=1175317 | chart=Kendrick Lamar - King Kunta (Horiiizon) [Genjuro's Kantan]
19:43:09 | INFO | Processing 40 / 290 | folder_id=1249951 | chart=Kobaryo - New Game Plus (Genjuro) [Challenge]
19:43:25 | INFO | Processing 60 / 290 | folder_id=1292034 | chart=Chroma - Sayonara Planet Wars (Mid) [2199's Oni]
19:43:49 | INFO | Processing 80 / 290 | folder_id=1656103 | chart=Chroma - fly in the galaxy (ikin5050) [Muzukashii]
19:44:08 | INFO | Processing 100 / 290 | folder_id=1966609 | chart=Shizuru (CV Nabatame Hitomi), Rino (CV Asumi Kana) - SUPER CHOCOLATE (Game Ver.) ([-E S I A-]) [MUZUKASHII]
19:

In [19]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd


# =============================================================================
# Config
# =============================================================================

DATA_ROOT = Path(r"D:\Study Abroad\course\DSCI498\Project\data")
CHART_INDEX_DIR = DATA_ROOT / "chart_index"
DATASET_DIR = DATA_ROOT / "beat_aligned_dataset"

CHART_SUMMARY_CSV = CHART_INDEX_DIR / "chart_build_summary.csv"
SEQUENCE_METADATA_CSV = DATASET_DIR / "sequence_metadata.csv"
TOKEN_JSON_DIR = DATASET_DIR / "token_json"
AUDIO_NPZ_DIR = DATASET_DIR / "audio_npz"

OUTPUT_SUMMARY_CSV = DATASET_DIR / "sanity_check_summary.csv"
OUTPUT_TOKEN_FREQ_CSV = DATASET_DIR / "sanity_check_token_frequency.csv"


# =============================================================================
# Helper functions
# =============================================================================

def require_file(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def load_chart_summary() -> pd.DataFrame:
    require_file(CHART_SUMMARY_CSV, "chart_build_summary.csv")
    return pd.read_csv(CHART_SUMMARY_CSV)


def load_sequence_metadata() -> pd.DataFrame:
    require_file(SEQUENCE_METADATA_CSV, "sequence_metadata.csv")
    return pd.read_csv(SEQUENCE_METADATA_CSV)


def find_json_column(df: pd.DataFrame) -> str:
    """
    Try to find the column in sequence_metadata that points to token JSON files.
    Adjust this if your actual column name differs.
    """
    candidate_cols = [
        "token_json_path",
        "tokens_json_path",
        "token_path",
        "tokens_path",
        "json_path",
    ]
    for col in candidate_cols:
        if col in df.columns:
            return col
    raise ValueError(
        f"Could not find token JSON path column in sequence_metadata.csv. "
        f"Available columns: {list(df.columns)}"
    )


def find_audio_column(df: pd.DataFrame) -> str:
    """
    Try to find the column in sequence_metadata that points to audio NPZ files.
    Adjust this if your actual column name differs.
    """
    candidate_cols = [
        "audio_npz_path",
        "audio_path",
        "npz_path",
    ]
    for col in candidate_cols:
        if col in df.columns:
            return col
    raise ValueError(
        f"Could not find audio NPZ path column in sequence_metadata.csv. "
        f"Available columns: {list(df.columns)}"
    )


def load_token_json(token_json_path: Path):
    require_file(token_json_path, "token_json")
    with open(token_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def extract_sequence_tokens(token_json_obj):
    """
    Normalize several possible JSON structures into a list of token lists.

    Expected output:
        List[List[str]]
    """
    if isinstance(token_json_obj, list):
        # Case 1: already a list of sequences
        if len(token_json_obj) == 0:
            return []
        if isinstance(token_json_obj[0], list):
            return token_json_obj
        if isinstance(token_json_obj[0], dict):
            # e.g. [{"seq_idx":0, "tokens":[...]}, ...]
            if "tokens" in token_json_obj[0]:
                return [x.get("tokens", []) for x in token_json_obj]
        raise ValueError("Unsupported list-based token JSON structure")

    if isinstance(token_json_obj, dict):
        # Case 2: {"sequences":[...]}
        if "sequences" in token_json_obj:
            seqs = token_json_obj["sequences"]
            if len(seqs) == 0:
                return []
            if isinstance(seqs[0], list):
                return seqs
            if isinstance(seqs[0], dict) and "tokens" in seqs[0]:
                return [x.get("tokens", []) for x in seqs]
            raise ValueError("Unsupported dict['sequences'] token JSON structure")

        # Case 3: {"sequence_token_data":[...]}
        if "sequence_token_data" in token_json_obj:
            seqs = token_json_obj["sequence_token_data"]
            if len(seqs) == 0:
                return []
            if isinstance(seqs[0], dict) and "tokens" in seqs[0]:
                return [x.get("tokens", []) for x in seqs]
            raise ValueError("Unsupported dict['sequence_token_data'] structure")

    raise ValueError("Unsupported token JSON format")


def load_audio_npz(audio_npz_path: Path) -> np.ndarray:
    require_file(audio_npz_path, "audio_npz")
    obj = np.load(audio_npz_path, allow_pickle=False)

    # Try common keys first
    for key in ["audio_sequences", "arr_0", "X", "data"]:
        if key in obj.files:
            return obj[key]

    if len(obj.files) == 1:
        return obj[obj.files[0]]

    raise ValueError(f"Could not identify audio array key in {audio_npz_path}. Keys: {obj.files}")


# =============================================================================
# Main sanity check
# =============================================================================

def main():
    print("Loading chart summary and sequence metadata...")
    chart_df = load_chart_summary()
    seq_df = load_sequence_metadata()

    print(f"Charts in chart_build_summary.csv: {len(chart_df)}")
    print(f"Rows in sequence_metadata.csv: {len(seq_df)}")

    if "status" in chart_df.columns:
        print("\nChart build status:")
        print(chart_df["status"].value_counts(dropna=False))

    # -------------------------------------------------------------------------
    # Basic chart-level sanity
    # -------------------------------------------------------------------------
    succeeded_charts = chart_df
    if "status" in chart_df.columns:
        succeeded_charts = chart_df[chart_df["status"] == "ok"].copy()

    print(f"\nSucceeded charts: {len(succeeded_charts)}")

    # -------------------------------------------------------------------------
    # Sequence metadata sanity
    # -------------------------------------------------------------------------
    print("\nSequence metadata columns:")
    print(list(seq_df.columns))

    # Sequence count distribution per chart if possible
    chart_id_cols = [c for c in ["folder_id", "chart_base", "chart_id"] if c in seq_df.columns]
    if len(chart_id_cols) >= 1:
        group_cols = chart_id_cols[:2] if len(chart_id_cols) >= 2 else chart_id_cols[:1]
        seq_count_per_chart = seq_df.groupby(group_cols).size().reset_index(name="n_sequences")
        print("\nSequence count per chart summary:")
        print(seq_count_per_chart["n_sequences"].describe())

    # -------------------------------------------------------------------------
    # Load token JSONs
    # -------------------------------------------------------------------------
    token_col = find_json_column(seq_df)
    print(f"\nUsing token JSON column: {token_col}")

    token_counter = Counter()
    token_lengths = []
    empty_sequence_count = 0
    total_sequences_from_json = 0
    token_json_failures = []

    # To avoid reading the same file repeatedly, deduplicate file paths
    unique_token_paths = seq_df[token_col].dropna().astype(str).unique().tolist()

    for i, token_path_str in enumerate(unique_token_paths, start=1):
        token_path = Path(token_path_str)
        try:
            token_json_obj = load_token_json(token_path)
            seq_tokens_list = extract_sequence_tokens(token_json_obj)

            total_sequences_from_json += len(seq_tokens_list)

            for tokens in seq_tokens_list:
                token_lengths.append(len(tokens))
                if len(tokens) == 0:
                    empty_sequence_count += 1
                token_counter.update(tokens)

        except Exception as e:
            token_json_failures.append({
                "token_json_path": str(token_path),
                "error": str(e),
            })

        if i % 50 == 0 or i == len(unique_token_paths):
            print(f"Processed token JSONs: {i} / {len(unique_token_paths)}")

    vocab = sorted(token_counter.keys())
    vocab_size = len(vocab)

    print("\nToken sanity:")
    print(f"Unique token JSON files: {len(unique_token_paths)}")
    print(f"Total sequences from token JSONs: {total_sequences_from_json}")
    print(f"Vocabulary size: {vocab_size}")
    print(f"Empty sequences: {empty_sequence_count}")
    print(f"Token JSON failures: {len(token_json_failures)}")

    if token_lengths:
        token_len_series = pd.Series(token_lengths)
        print("\nToken length summary:")
        print(token_len_series.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

    # -------------------------------------------------------------------------
    # Inspect audio arrays
    # -------------------------------------------------------------------------
    audio_col = find_audio_column(seq_df)
    print(f"\nUsing audio NPZ column: {audio_col}")

    audio_shapes = []
    audio_nan_files = 0
    audio_inf_files = 0
    audio_failures = []
    total_sequences_from_audio = 0

    unique_audio_paths = seq_df[audio_col].dropna().astype(str).unique().tolist()

    for i, audio_path_str in enumerate(unique_audio_paths, start=1):
        audio_path = Path(audio_path_str)
        try:
            arr = load_audio_npz(audio_path)

            audio_shapes.append(tuple(arr.shape))
            total_sequences_from_audio += int(arr.shape[0])

            if np.isnan(arr).any():
                audio_nan_files += 1
            if np.isinf(arr).any():
                audio_inf_files += 1

        except Exception as e:
            audio_failures.append({
                "audio_npz_path": str(audio_path),
                "error": str(e),
            })

        if i % 50 == 0 or i == len(unique_audio_paths):
            print(f"Processed audio NPZs: {i} / {len(unique_audio_paths)}")

    print("\nAudio sanity:")
    print(f"Unique audio NPZ files: {len(unique_audio_paths)}")
    print(f"Total sequences from audio arrays: {total_sequences_from_audio}")
    print(f"Distinct audio shapes: {sorted(set(audio_shapes))[:10]}")
    print(f"Audio files with NaN: {audio_nan_files}")
    print(f"Audio files with Inf: {audio_inf_files}")
    print(f"Audio NPZ failures: {len(audio_failures)}")

    # -------------------------------------------------------------------------
    # Cross-check counts
    # -------------------------------------------------------------------------
    print("\nCross-check:")
    print(f"sequence_metadata rows         = {len(seq_df)}")
    print(f"total sequences from token JSON = {total_sequences_from_json}")
    print(f"total sequences from audio NPZ  = {total_sequences_from_audio}")

    # -------------------------------------------------------------------------
    # Save summaries
    # -------------------------------------------------------------------------
    summary_rows = []

    summary_rows.append({"metric": "n_chart_rows", "value": len(chart_df)})
    summary_rows.append({"metric": "n_succeeded_charts", "value": len(succeeded_charts)})
    summary_rows.append({"metric": "n_sequence_metadata_rows", "value": len(seq_df)})
    summary_rows.append({"metric": "n_unique_token_json_files", "value": len(unique_token_paths)})
    summary_rows.append({"metric": "n_unique_audio_npz_files", "value": len(unique_audio_paths)})
    summary_rows.append({"metric": "n_total_sequences_from_token_json", "value": total_sequences_from_json})
    summary_rows.append({"metric": "n_total_sequences_from_audio_npz", "value": total_sequences_from_audio})
    summary_rows.append({"metric": "vocab_size", "value": vocab_size})
    summary_rows.append({"metric": "empty_sequence_count", "value": empty_sequence_count})
    summary_rows.append({"metric": "token_json_failures", "value": len(token_json_failures)})
    summary_rows.append({"metric": "audio_npz_failures", "value": len(audio_failures)})
    summary_rows.append({"metric": "audio_nan_files", "value": audio_nan_files})
    summary_rows.append({"metric": "audio_inf_files", "value": audio_inf_files})

    if token_lengths:
        summary_rows.extend([
            {"metric": "token_len_min", "value": int(np.min(token_lengths))},
            {"metric": "token_len_mean", "value": float(np.mean(token_lengths))},
            {"metric": "token_len_median", "value": float(np.median(token_lengths))},
            {"metric": "token_len_p90", "value": float(np.percentile(token_lengths, 90))},
            {"metric": "token_len_p95", "value": float(np.percentile(token_lengths, 95))},
            {"metric": "token_len_p99", "value": float(np.percentile(token_lengths, 99))},
            {"metric": "token_len_max", "value": int(np.max(token_lengths))},
        ])

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False, encoding="utf-8-sig")

    token_freq_df = pd.DataFrame(
        [{"token": token, "count": count} for token, count in token_counter.most_common()]
    )
    token_freq_df.to_csv(OUTPUT_TOKEN_FREQ_CSV, index=False, encoding="utf-8-sig")

    print("\nSaved:")
    print(OUTPUT_SUMMARY_CSV)
    print(OUTPUT_TOKEN_FREQ_CSV)

    # -------------------------------------------------------------------------
    # Optional: show suspicious cases
    # -------------------------------------------------------------------------
    if token_lengths:
        suspicious_long_threshold = np.percentile(token_lengths, 99)
        print(f"\nSuggested suspicious token length threshold (p99): {suspicious_long_threshold:.2f}")

    if token_json_failures:
        print("\nToken JSON failures (first 10):")
        print(pd.DataFrame(token_json_failures).head(10))

    if audio_failures:
        print("\nAudio NPZ failures (first 10):")
        print(pd.DataFrame(audio_failures).head(10))


if __name__ == "__main__":
    main()

Loading chart summary and sequence metadata...
Charts in chart_build_summary.csv: 290
Rows in sequence_metadata.csv: 31119

Chart build status:
status
ok    290
Name: count, dtype: int64

Succeeded charts: 290

Sequence metadata columns:
['chart_id', 'folder_id', 'chart_base', 'seq_idx', 'start_frame', 'end_frame', 'n_events', 'n_tokens', 'audio_npz_path', 'token_json_path']

Sequence count per chart summary:
count    290.000000
mean     107.306897
std       63.582649
min       19.000000
25%       64.250000
50%       89.500000
75%      146.000000
max      364.000000
Name: n_sequences, dtype: float64

Using token JSON column: token_json_path
Processed token JSONs: 50 / 290
Processed token JSONs: 100 / 290
Processed token JSONs: 150 / 290
Processed token JSONs: 200 / 290
Processed token JSONs: 250 / 290
Processed token JSONs: 290 / 290

Token sanity:
Unique token JSON files: 290
Total sequences from token JSONs: 31119
Vocabulary size: 82
Empty sequences: 2341
Token JSON failures: 0

Toke